# 04. Error analysis

Notebook para documentar falsos positivos, falsos negativos y patrones de error.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PROCESSED = BASE_DIR / "data" / "processed"
OUTPUTS_TABLES = BASE_DIR / "outputs" / "tables"
OUTPUTS_FIGURES = BASE_DIR / "outputs" / "figures"
OUTPUTS_METRICS = BASE_DIR / "outputs" / "metrics"

OUTPUTS_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUTS_FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUTS_METRICS.mkdir(parents=True, exist_ok=True)



In [ ]:
import joblib

test_df = pd.read_csv(DATA_PROCESSED / "test.csv")
ext_df = pd.read_csv(DATA_PROCESSED / "external_validation_detoxis.csv")
baseline_model = joblib.load(BASE_DIR / "models" / "artifacts" / "baseline_tfidf_logreg.joblib")


In [ ]:
test_df["baseline_pred"] = baseline_model.predict(test_df["text_clean"])
ext_df["baseline_pred"] = baseline_model.predict(ext_df["text_clean"])

test_df["error_type"] = np.where(
    test_df["label"] == test_df["baseline_pred"],
    "correcto",
    np.where(test_df["label"] == 1, "FN", "FP")
)

ext_df["error_type"] = np.where(
    ext_df["label"] == ext_df["baseline_pred"],
    "correcto",
    np.where(ext_df["label"] == 1, "FN", "FP")
)

test_df["error_type"].value_counts(), ext_df["error_type"].value_counts()


In [ ]:
errors_test = test_df.loc[test_df["error_type"] != "correcto", ["text_clean","label","baseline_pred","error_type","dataset_source"]].head(50)
errors_ext = ext_df.loc[ext_df["error_type"] != "correcto", ["text_clean","label","baseline_pred","error_type","dataset_source"]].head(50)

errors_test.to_csv(OUTPUTS_TABLES / "baseline_errors_test.csv", index=False, encoding="utf-8-sig")
errors_ext.to_csv(OUTPUTS_TABLES / "baseline_errors_external.csv", index=False, encoding="utf-8-sig")

errors_test.head(10)


## Nota

Cuando el transformer termine de entrenar, repetir aquí el mismo análisis para comparar errores entre baseline y BETO.


## Conclusiones

Completar aquí los patrones hallados: sarcasmo, ambigüedad, grosería sin grupo objetivo, etc.
